#### using the interconnections and splitting tasks

In [1]:
# runningmax abstraction with PAC guarantee example

# needed libraries
import numpy as np
import scipy.special as sp
import time
from itertools import product
import random
import gurobipy as gp
from gurobipy import GRB
from joblib import Parallel, delayed
from scipy.optimize import fsolve
from math import comb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import scipy.stats as stats
from scipy.optimize import brentq
from scipy.stats import truncnorm

In [2]:
N_subsys = 80 # number of subsystem considered
# for the subsystem
# choice of N for SOP
N_j = 1000 # j=1,...,N_subsys
n_dim = 1 # dimension of subsystem state set 
N_pos = 500 # N used for computing post in subsystem abstraction

# system dynamics
eta_x1, eta_w1 = 0.5, 0.5
eta_x = np.array([eta_x1]) # discretization vector
eta_ww = np.array([eta_w1 for i in range(N_subsys)])
# set of states
a1, a2 = 0, 32
a = np.array([a1])  # lower bounds
b = np.array([a2])  # upper bounds

dim1_hat = np.arange(a1+eta_x1, a2-eta_x1, eta_x1)
X_ha = np.array(list(product(dim1_hat))) 
X_haa = np.around(X_ha, 2)

dim1_hat_w = np.arange(a1+eta_w1, a2-eta_w1, eta_w1)

# Sample N_j i.i.d. pairs (x_i, x_hat_i)
x_samples = np.random.uniform(low=a, high=b, size=(N_j,1))
x_hat_indices = np.random.choice(len(X_haa), size=N_j, replace=True)
x_hat_samples = X_haa[x_hat_indices]

# Combine into a list of tuples or array of pairs
X_pairs = list(zip(x_samples, x_hat_samples))
print("susbsystem samples: ",len(X_pairs))

susbsystem samples:  1000


In [5]:
# system dynamics as black-box simulator
def sys_dyn(x,u,w):
    f_xt1 = min(0.75*(x[0] + u), w + 1, 32)
    nxt = [max(min(f_xt1, a2), a1)]
    nxt = [round(x, 2) for x in nxt]
    return nxt
        
def M_dyn(x): # intercon dynamics
    return max(x)

def generate_grid_centers(a, b, N_pos):
    dim = len(a)
    # Estimate number of points per dimension (evenly)
    k = int(np.round(N_pos ** (1 / dim)))
    total_points = k ** dim

    # Generate grid centers per dimension
    grid_axes = []
    for ai, bi in zip(a, b):
        step = (bi - ai) / k
        centers = ai + (np.arange(k) + 0.5) * step
        grid_axes.append(centers)

    # Cartesian product of all centers (i.e., subgrid centers)
    all_points = np.array(list(product(*grid_axes)))

    # Trim if more than needed
    if total_points > N_pos:
        all_points = all_points[:N_pos]

    return all_points

def Q_w(w, eta_w=eta_w1, w_min=0.0, w_max=32.0): # quantizer of internal inputs W
    w_hat = eta_w * np.round(w / eta_w)
    return np.clip(w_hat, w_min, w_max)

# for abstract subsystem
def sys_dyn_hat(y, u, w):
    y = np.asarray(y)
    X_hat_arra = np.array(X_haa)  
    a, b = y-eta_x/2, y+eta_x/2 
    # sampling N points as subgrid centres for each cell to obtain an estimate of the reachable sets as done in the paper
    sampled_points = generate_grid_centers(a, b, N_pos)
    
    # Compute successors
    successors = np.array([sys_dyn(x, u, w) for x in sampled_points])

    # Compute mean and max distance
    m = np.mean(successors, axis=0)
    r = np.max(np.abs(successors - m), axis=0)

    # Find points in X_hat_array within the under-approximation of reachable sets
    mask = np.all(np.abs(X_hat_arra - m) <= (r + eta_x / 2), axis=1)

    return X_hat_arra[mask]

# set of inputs
U, W = range(7+1), range(32+1) # subsystems external and internal input sets 
M = len(U)
U_array = np.array(U)
print("input number: ",M)

ubpr = 100
lbp, ubp = -100, 100
lbpc, ubpc = -100, 100
lbe, ube = -100, 0 

input number:  8


In [6]:
## compositional SOP
m1 = gp.Model("PAC_ASF_compositional")

vartheta = m1.addVar(vtype = GRB.CONTINUOUS, name = "vartheta", lb = lbe, ub = ube)

eps_vars = 1/len(X_pairs) 
m1.update()

gamma_b = 1e-4 # 1e-4 
delta_b = gamma_b * np.linalg.norm(np.array(eta_x), ord=np.inf) **2 # ||eta_x||^2 * gamma
tau_b = 3.001 # this is \rho in the paper
print("delta: ", delta_b)
print("epsilon: ", (delta_b/gamma_b)**0.5)

num_variables = 6 # 3  # number of coefficients as a result of the chosen degree of asf
variable_names = [f"lambda_{i}" for i in range(1, num_variables + 1)]
variable_objs = {name_i: m1.addVar(vtype=GRB.CONTINUOUS, name=name_i, lb=lbpc, ub=ubpc) for name_i in variable_names}

def Asf(x, x_hat):
    # Coefficients from the declared variables
    # c1,c2,c3 = variable_objs.values()
    c1,c2,c3,c4,c5,c6 = variable_objs.values()
    x, y = x[0], x_hat[0]
    # Polynomial expression using the variables
    result = (
        # c1 + c2*x + c3*y
        c1 + c2*x + c3*y + c4*x*y + c5*x**2 +c6*y**2 
    )
    return result

# Define lambda functions for efficiency
max_abs_diff = lambda x, y: gamma_b * np.max(np.abs(x - y))**2
asf_value = lambda x, y: Asf(x, y)

# First ASF condition subsystem level
t_init = time.time()
sum1 = gp.quicksum(eps_vars * (max_abs_diff(x, y) - asf_value(x, y)) for (x,y) in X_pairs)
m1.addConstr(sum1 <= vartheta)
t_fin = time.time()
diff_t = t_fin - t_init
print("done1: ",diff_t)
m1.update()

t_init = time.time()
# ASF 2 Enforce sum2 - Asf <= vartheta

def compute_sys_dyn(l, i, r):
    return l, i, r, sys_dyn_hat(X_pairs[l][1], U[i], Q_w(W[r])), sys_dyn(X_pairs[l][0], U[i], W[r])

# Step 1: Compute system dynamics in parallel
num_jobs = -1  # Use all available cores
dyn_results = Parallel(n_jobs=num_jobs, verbose=2)(
    delayed(compute_sys_dyn)(l, i, r) for l in range(len(X_pairs)) for i in range(len(U)) for r in range(len(W))
)

# Step 2: Add constraints sequentially (Gurobi constraint addition must be sequential during parallelization)
for l, i, r, dyn_hat_output, dyn_output in dyn_results:
    m1.addConstr(
        gp.quicksum(1/(len(dyn_hat_output)) * asf_value(dyn_output, yn) for yn in dyn_hat_output) # sigma taken uniformly
        - tau_b*asf_value(X_pairs[l][0], X_pairs[l][1]) + (tau_b-1)*delta_b <= vartheta
    )

t_fin = time.time()
diff_t = t_fin - t_init
m1.update()
print("done2: ",diff_t)

m1.Params.LogToConsole = 0
m1.setParam("NumericFocus", 3)
m1.setParam('NonConvex', 2)

m1.update()
print("Start now")
print("Number of variables:", m1.numVars)
print("Number of constraints:", m1.numConstrs)
t_init = time.time()

m1.setObjective(vartheta, GRB.MINIMIZE)
m1.setParam('Presolve', 0)

m1.optimize()
t_fin = time.time()
diff_t = t_fin - t_init
print("opt_status:", m1.status)

m1.write("comp_with_PAC_Infeasible.lp")

# Check if optimization was successful
if m1.status == GRB.OPTIMAL:
    with open('comp_with_PAC_variables.txt', 'w') as file:
        for v in m1.getVars():
            file.write(f'{v.VarName} {v.X}\n')
elif m1.status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    m1.computeIIS()
    m1.write("comp_with_PAC_Infeasible.ilp")
    print("IIS written to subsys_with_PAC_Infeasible.ilp")

    with open("comp_with_PAC_Infeasible.txt", "w") as f:
        for c in m1.getConstrs():
            if c.IISConstr:
                f.write(f"Infeasible constraint: {c.ConstrName}\n")
        for v in m1.getVars():
            if v.IISLB or v.IISUB:
                f.write(f"Infeasible bound on variable: {v.VarName}\n")

else:
    print("Optimization was not successful (not optimal or infeasible). Status:", m1.status)

# Count binding constraints
sup_const = [constr for constr in m1.getConstrs() if abs(constr.Slack) < 1e-4]
s_ = len(sup_const)
print(f"Number of constraints within tol: {s_}")
print("Time (in s):", diff_t)

Set parameter Username
Set parameter LicenseID to value 2778218
Academic license - for non-commercial use only - expires 2027-02-11
delta:  2.5e-05
epsilon:  0.5
done1:  0.029542922973632812


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    0.9s
[Parallel(n_jobs=-1)]: Done 142 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done 345 tasks      | elapsed:    4.4s
[Parallel(n_jobs=-1)]: Done 628 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-1)]: Done 993 tasks      | elapsed:   11.4s
[Parallel(n_jobs=-1)]: Done 1438 tasks      | elapsed:   16.2s
[Parallel(n_jobs=-1)]: Done 1965 tasks      | elapsed:   21.9s
[Parallel(n_jobs=-1)]: Done 2572 tasks      | elapsed:   28.5s
[Parallel(n_jobs=-1)]: Done 3261 tasks      | elapsed:   35.9s
[Parallel(n_jobs=-1)]: Done 4030 tasks      | elapsed:   44.3s
[Parallel(n_jobs=-1)]: Done 4881 tasks      | elapsed:   53.5s
[Parallel(n_jobs=-1)]: Done 5812 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 6825 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 7918 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 9093 tasks      | 

done2:  2978.1164689064026
Start now
Number of variables: 7
Number of constraints: 264001
opt_status: 2
Number of constraints within tol: 52
Time (in s): 0.2355659008026123


In [8]:
# Count binding constraints
def prune_constraints_inf_norm(m, tol=1e-4):
    """
    Iteratively test which constraints are necessary
    based on their effect on the optimal solution.
    Uses infinity norm for solution difference.
    Returns the final set of constraints deemed necessary.
    """
    # Solve original model
    m.optimize()
    if m.status != gp.GRB.OPTIMAL:
        raise RuntimeError("Original model is not optimal.")
    
    # Extract original solution sol*
    sol_star = np.array([v.X for v in m.getVars()])
    
    # Initial set of constraints
    C = list(m.getConstrs())
    
    necessary_constraints = []
    
    for constr in C:
        # Create a clone of the model
        m_copy = m.copy()
        
        # Remove the constraint from the copy
        constr_copy = m_copy.getConstrByName(constr.ConstrName)
        m_copy.remove(constr_copy)
        m_copy.update()
        
        # Solve the reduced model
        m_copy.optimize()
        
        if m_copy.status == gp.GRB.OPTIMAL:
            sol_starstar = np.array([v.X for v in m_copy.getVars()])
            diff = np.linalg.norm(sol_star - sol_starstar, ord=np.inf)  # infinity norm
            
            # If the solution shifts a lot, keep the constraint
            if diff > tol:
                necessary_constraints.append(constr)
        # else:
        #     # If infeasible or unbounded without this constraint, it’s definitely necessary
        #     necessary_constraints.append(constr)
    
    return necessary_constraints

In [ ]:
# Usage
s_N_comp = len(prune_constraints_inf_norm(m1, tol=1e-4)) # for subsys
print(f"Number of support constraints: {s_N_comp}")

#### nonconvex SOP PAC bound
Consider Equation (7) in https://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=8299432

#### comparison of $\beta,\alpha$, and $\mathcal{N}$.

In [7]:
# as alternative, consider eqn 7 in the paper
def epsil(k,N1,beta1):
    res = (beta1/(N1*comb(N1, k)))**(1/(N1-k))
    return 1-res

In [20]:
beta_subsys = 10**-6
# alpha_sN_subsys = epsil(s_N_comp, N_j, beta_subsys)
alpha_sN_subsys = epsil(1, 5000, beta_subsys)

print(f"With confidence of {(1-beta_subsys)*100:.6f}%, the non-violation of at least {1-alpha_sN_subsys:.6f}, and violation {alpha_sN_subsys:.6f}")

With confidence of 99.999900%, the non-violation of at least 0.993848, and violation 0.006152


In [12]:
N_subsys, N_0, N_j = 4, 1000, 1000
N_bar = N_0 + N_subsys*N_j
alpha_bar = (N_subsys + 1)*alpha_sN_subsys
beta_bar = (N_subsys + 1)*beta_subsys
print("Compositional result")
print(f"Compositional samples: {(N_bar):.6f}")
print(f"With confidence of {(1-beta_bar)*100:.6f}%, the non-violation of at least {1-alpha_bar:.6f}, and violation {alpha_bar:.6f}")

Compositional result
Compositional samples: 5000.000000
With confidence of 99.999500%, the non-violation of at least 0.863602, and violation 0.136398


In [9]:
def epsil_nonconvex(k, N1, beta1):
    """
    Compute PAC violation probability epsilon for the non-convex scenario bound.
    
    Solves:
        sum_{i=0}^k C(N1,i) * eps^i * (1-eps)^(N1-i) = beta1
    """

    def f(eps):
        s = 0.0
        for i in range(k+1):
            s += comb(N1, i) * (eps**i) * ((1-eps)**(N1-i))
        return s - beta1

    # epsilon must lie in (0,1)
    eps = brentq(f, 1e-12, 1-1e-12)
    
    return eps


In [27]:
beta_subsys = 10**-6
N_j = 9000
# alpha_sN_subsys = epsil(s_N_comp, N_j, beta_subsys)
alpha_sN_subsys = epsil_nonconvex(1, N_j, beta_subsys)

print(f"With confidence of {(1-beta_subsys)*100:.6f}%, the non-violation of at least {1-alpha_sN_subsys:.6f}, and violation {alpha_sN_subsys:.6f}")

With confidence of 99.999900%, the non-violation of at least 0.998147, and violation 0.001853


In [28]:
N_subsys = 80
N_bar = N_subsys*N_j
alpha_bar = (N_subsys)*alpha_sN_subsys
beta_bar = (N_subsys)*beta_subsys
print("Compositional result")
print(f"Compositional samples: {(N_bar):.6f}")
print(f"With confidence of {(1-beta_bar)*100:.6f}%, the non-violation of at least {1-alpha_bar:.6f}, and violation {alpha_bar:.6f}")

Compositional result
Compositional samples: 720000.000000
With confidence of 99.992000%, the non-violation of at least 0.851788, and violation 0.148212
